In [2]:
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional, Tuple
import pickle
import hashlib

from typing import List, Dict, Optional
from datetime import datetime
import re
import time
import torch

In [3]:
os.environ['HF_TOKEN'] = 'your_token_here'  # 可以随便填，不验证

# 或者设置离线模式
os.environ['HF_HUB_OFFLINE'] = '1'  # 完全离线模式

In [4]:
class FAISSMenuStore:
    """基于FAISS的菜单向量存储"""
    
    def __init__(self, data_path: str = "./data/menu.json", embedding_dim: int = 384):
        """
        初始化FAISS存储
        
        Args:
            data_path: 菜单JSON路径
            embedding_dim: 向量维度（all-MiniLM-L6-v2是384维）
        """
        print("\n🔧 初始化FAISS向量存储...")
        
        self.data_path = data_path
        self.embedding_dim = embedding_dim
        self.index_path = "./data/faiss_index.bin"
        self.metadata_path = "./data/faiss_metadata.pkl"
        
        # 1. 初始化embedding模型
        print("  加载embedding模型...")
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        print(f"  ✅ 模型加载完成，维度: {self.embedder.get_sentence_embedding_dimension()}")
        
        # 2. 初始化或加载FAISS索引
        self._init_index()
        
        # 3. 加载菜单数据
        self.menu_items = self._load_menu()
        self.id_to_item = {item['id']: item for item in self.menu_items}
        
        print(f"  📊 共加载 {len(self.menu_items)} 个菜品")

        # 开始菜品转向量
        print('开始菜品转向量')
        self.build_index()
    
    def _init_index(self):
        """初始化FAISS索引"""
        if os.path.exists(self.index_path):
            # 加载已有索引
            print("  加载已有FAISS索引...")
            self.index = faiss.read_index(self.index_path)
            with open(self.metadata_path, 'rb') as f:
                self.id_to_index, self.index_to_id = pickle.load(f)
            print(f"  ✅ 加载完成，索引大小: {self.index.ntotal}")
        else:
            # 创建新索引
            print("  创建新FAISS索引...")
            # 使用L2距离的索引
            self.index = faiss.IndexFlatL2(self.embedding_dim)
            self.id_to_index = {}  # 菜品ID到索引位置的映射
            self.index_to_id = []  # 索引位置到菜品ID的映射
            print("  ✅ 新索引创建完成")
            return
    
    def _load_menu(self) -> List[Dict]:
        """加载菜单数据"""
        if not os.path.exists(self.data_path):
            print(f"  ⚠️ 菜单文件不存在，创建示例菜单...")
        
        with open(self.data_path, 'r', encoding='utf-8') as f:
            menu = json.load(f)
        return menu["dishes"]
    
    def _get_dish_text(self, dish: Dict) -> str:
        """获取菜品的文本表示（用于向量化）"""
        return f"""
        菜品：{dish['name']}
        价格：{dish['price']}元
        描述：{dish['description']}
        配料：{', '.join(dish['ingredients'])}
        辣度：{dish['spicy_level']}
        类别：{dish['category_name']}
        标签：{', '.join(dish['tags'])}
        """
    
    def build_index(self):
        """构建FAISS索引"""
        print("\n🔨 构建FAISS索引...")
        
        if self.index.ntotal > 0:
            print(f"  索引已存在，大小: {self.index.ntotal}")
            return
        
        # 为每个菜品生成向量
        embeddings = []
        
        for dish in self.menu_items:
            text = self._get_dish_text(dish)
            embedding = self.embedder.encode(text)
            embeddings.append(embedding)
            
            # 记录映射
            idx = len(self.index_to_id)
            self.id_to_index[dish['id']] = idx
            self.index_to_id.append(dish['id'])
        
        # 转换为numpy数组
        embeddings_array = np.array(embeddings).astype('float32')
        
        # 添加到FAISS索引
        self.index.add(embeddings_array)
        
        # 保存索引
        faiss.write_index(self.index, self.index_path)
        with open(self.metadata_path, 'wb') as f:
            pickle.dump((self.id_to_index, self.index_to_id), f)
        
        print(f"  ✅ 索引构建完成，共 {self.index.ntotal} 个向量")
        print(f"  💾 索引已保存到 {self.index_path}")
    
    def search(self, query: str, k: int = 5) -> List[Dict]:
        """
        搜索相似菜品
        
        Args:
            query: 查询文本
            k: 返回结果数量
            
        Returns:
            相似菜品列表
        """
        # 生成查询向量
        query_embedding = self.embedder.encode(query).astype('float32').reshape(1, -1)
        
        # 搜索
        distances, indices = self.index.search(query_embedding, k)
        
        results = []
        for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            if idx == -1:  # FAISS可能返回-1
                continue
            
            dish_id = self.index_to_id[idx]
            dish = self.id_to_item[dish_id]
            
            # 计算相似度分数（距离越小越相似）
            similarity = 1 / (1 + dist)
            
            results.append({
                "rank": i + 1,
                "id": dish_id,
                "name": dish['name'],
                "price": dish['price'],
                "description": dish['description'],
                "spicy_level": dish['spicy_level'],
                "category": dish['category_name'],
                "similarity": round(similarity, 3),
                "distance": float(dist)
            })
        
        return results
    
    def filter_search(self, query: str, filters: Dict, k: int = 10) -> List[Dict]:
        """
        带过滤条件的搜索
        
        Args:
            query: 查询文本
            filters: 过滤条件，如 {"max_price": 40, "category": "川菜"}
            k: 先检索更多结果再过滤
        """
        # 先搜索更多结果
        all_results = self.search(query, k=k*2)
        
        # 应用过滤
        filtered = []
        for dish in all_results:
            match = True
            
            # 价格上限过滤
            if "max_price" in filters and dish['price'] > filters['max_price']:
                match = False
            
            # 价格下限过滤
            if "min_price" in filters and dish['price'] < filters['min_price']:
                match = False
            
            # 类别过滤
            if "category" in filters and dish['category_name'] != filters['category']:
                match = False
            
            # 辣度过滤
            if "spicy_level" in filters and dish['spicy_level'] != filters['spicy_level']:
                match = False
            
            if match:
                filtered.append(dish)
        
        return filtered[:k]
    
    def get_by_category(self, category: str) -> List[Dict]:
        """按类别获取菜品"""
        return [d for d in self.menu_items if d['category'] == category]
    
    def get_by_price_range(self, min_price: float, max_price: float) -> List[Dict]:
        """按价格范围获取菜品"""
        return [d for d in self.menu_items if min_price <= d['price'] <= max_price]

rerank

In [5]:
class HybridRerankStore:
    """混合重排序 - 规则 + 模型"""
    
    def __init__(self, vectorDB):
        self.vector_store = vectorDB
        
        # 加载轻量级rerank模型
        try:
            from sentence_transformers import CrossEncoder
            # 英文菜单 self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
            self.reranker = CrossEncoder(
                'BAAI/bge-reranker-base',  # 中文基础版
                # 'BAAI/bge-reranker-large',  # 中文大模型（更好但更慢）
                max_length=512,
                device='cuda' if torch.cuda.is_available() else 'cpu'
            )
            self.use_model = True
            print("rerank模型加载完成")
        except Exception as e:
            print(f"❌ 加载rerank模型失败: {e}")
            self.use_model = False
            print("使用规则重排序")
    
    def _rule_score(self, dish: dict, query: str) -> float:
        """规则分数"""
        score = 0
        query = query.lower()
        
        # 辣度匹配
        if "辣" in query:
            spicy_map = {"重辣": 1.0, "中辣": 0.8, "微辣": 0.5, "不辣": 0.0}
            score += spicy_map.get(dish['spicy_level'], 0) * 0.3
        
        # 价格匹配
        if "便宜" in query and dish['price'] < 30:
            score += 0.2
        if "贵" in query and dish['price'] > 50:
            score += 0.2
        
        # 菜品名匹配
        if dish['name'] in query:
            score += 0.5
        
        return score
    
    def search(self, query: str, k: int = 5):
        """搜索 + 混合Rerank"""
        
        # 1. 获取候选
        candidates = self.vector_store.search(query, k=20)
        
        # 2. 计算分数
        for dish in candidates:
            # 向量相似度分数
            vec_score = dish['similarity']
            
            # 规则分数
            rule_score = self._rule_score(dish, query)
            
            # 模型分数（如果有）
            if self.use_model:
                model_score = self.reranker.predict([[query, dish['name']]])[0]
                # 混合分数
                dish['final_score'] = vec_score * 0.3 + rule_score * 0.3 + model_score * 0.4
            else:
                # 只有向量+规则
                dish['final_score'] = vec_score * 0.6 + rule_score * 0.4
        
        # 3. 排序
        results = sorted(candidates, key=lambda x: x['final_score'], reverse=True)
        
        return results[:k]

In [7]:
VectorDB = FAISSMenuStore()
rerankModel = HybridRerankStore(VectorDB)


🔧 初始化FAISS向量存储...
  加载embedding模型...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ 模型加载完成，维度: 384
  加载已有FAISS索引...
  ✅ 加载完成，索引大小: 20
  📊 共加载 20 个菜品
开始菜品转向量

🔨 构建FAISS索引...
  索引已存在，大小: 20


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


rerank模型加载完成


In [41]:
def test_build_index(store):
    """测试2: 构建索引"""
    print("测试2: 构建FAISS索引")
    
    if store.index.ntotal >= 0:
        print(f"📊 索引已存在，大小: {store.index.ntotal}")
        return True
    
    try:
        start_time = time.time()
        store.build_index()
        elapsed = time.time() - start_time
        
        print(f"✅ 索引构建完成!")
        print(f"📊 索引大小: {store.index.ntotal}")
        print(f"⏱️  耗时: {elapsed:.2f}秒")
        return True
    except Exception as e:
        print(f"❌ 构建索引失败: {e}")
        return False

test_build_index(VectorDB)        

测试2: 构建FAISS索引
📊 索引已存在，大小: 20


True

In [48]:
def test_basic_search(store):
    """测试3: 基础搜索"""
    print("测试3: 基础搜索功能")
    
    test_queries = [
        "辣的菜",
        "便宜的菜", 
        "清淡的菜",
        "麻婆豆腐",
        "下饭菜",
        "鸡肉",
        "鱼",
        "汤"
    ]
    
    all_passed = True
    
    for query in test_queries:
        print(f"\n🔍 搜索: '{query}'")
        
        try:
            start_time = time.time()
            results = store.search(query, k=10)
            elapsed = time.time() - start_time
            
            if results:
                print(f"✅ 找到 {len(results)} 个结果 (耗时: {elapsed*1000:.2f}ms)")
                for r in results:
                    # 相似度星星
                    stars = "★" * int(r['similarity'] * 5) + "☆" * (5 - int(r['similarity'] * 5))
                    spicy = "🌶️" * (["不辣", "微辣", "中辣", "重辣"].index(r['spicy_level']) + 1 if r['spicy_level'] in [ "微辣", "中辣", "重辣"] else 0)
                    
                    print(f"   {r['rank']}. {r['name']:15} {r['price']:3}元 {spicy:6} {stars} {r['similarity']:.1%}")
            else:
                print(f"⚠️  未找到结果")
                all_passed = False
                
        except Exception as e:
            print(f"❌ 搜索失败: {e}")
            all_passed = False
    
    return all_passed

test_basic_search(VectorDB)

测试3: 基础搜索功能

🔍 搜索: '辣的菜'
✅ 找到 10 个结果 (耗时: 11.95ms)
   1. 皮蛋瘦肉粥            16元        ★★☆☆☆ 46.2%
   2. 拔丝地瓜             28元        ★★☆☆☆ 45.8%
   3. 清蒸鲈鱼             68元        ★★☆☆☆ 45.8%
   4. 蒜蓉西兰花            22元        ★★☆☆☆ 44.6%
   5. 小笼包              22元        ★★☆☆☆ 44.6%
   6. 地三鲜              26元        ★★☆☆☆ 44.5%
   7. 红烧肉              52元        ★★☆☆☆ 44.3%
   8. 玉米排骨汤            38元        ★★☆☆☆ 44.2%
   9. 宫保鸡丁             42元 🌶️🌶️   ★★☆☆☆ 43.0%
   10. 西红柿炒鸡蛋           28元        ★★☆☆☆ 42.9%

🔍 搜索: '便宜的菜'
✅ 找到 10 个结果 (耗时: 10.74ms)
   1. 皮蛋瘦肉粥            16元        ★★☆☆☆ 46.3%
   2. 清蒸鲈鱼             68元        ★★☆☆☆ 45.6%
   3. 拔丝地瓜             28元        ★★☆☆☆ 45.5%
   4. 小笼包              22元        ★★☆☆☆ 44.5%
   5. 地三鲜              26元        ★★☆☆☆ 44.3%
   6. 玉米排骨汤            38元        ★★☆☆☆ 44.2%
   7. 红烧肉              52元        ★★☆☆☆ 44.1%
   8. 蒜蓉西兰花            22元        ★★☆☆☆ 44.0%
   9. 宫保鸡丁             42元 🌶️🌶️   ★★☆☆☆ 42.9%
   10. 酸菜鱼              58元 🌶️🌶️🌶️

True

In [8]:
def test_rerank_effect(store, rerank_store):
    """测试Rerank效果"""
    
    print("🚀 测试Rerank效果")
    print("=" * 60)
    
    # 1. 无Rerank
    print("\n📊 无Rerank:")
    results = store.search("辣的菜", k=10)
    # print(results)
    for r in results:
        print(f"  {r['rank']}. {r['name']} - {r['similarity']:.1%}")
    
    # 2. 有Rerank
    print("\n📊 有Rerank:")
    rerank_results = rerank_store.search("辣的菜", k=10)
    # print(rerank_results)
    for new_r in rerank_results:
        print(f"  {new_r['rank']}. {new_r['name']} - {new_r['similarity']:.1%}")

test_rerank_effect(VectorDB, rerankModel)

🚀 测试Rerank效果

📊 无Rerank:
  1. 皮蛋瘦肉粥 - 46.2%
  2. 拔丝地瓜 - 45.8%
  3. 清蒸鲈鱼 - 45.8%
  4. 蒜蓉西兰花 - 44.6%
  5. 小笼包 - 44.6%
  6. 地三鲜 - 44.5%
  7. 红烧肉 - 44.3%
  8. 玉米排骨汤 - 44.2%
  9. 宫保鸡丁 - 43.0%
  10. 西红柿炒鸡蛋 - 42.9%

📊 有Rerank:
  16. 酸辣土豆丝 - 41.7%
  18. 香辣盆盆虾 - 41.5%
  11. 酸菜鱼 - 42.6%
  14. 金牌麻婆豆腐 - 41.9%
  4. 蒜蓉西兰花 - 44.6%
  19. 干煸四季豆 - 40.9%
  10. 西红柿炒鸡蛋 - 42.9%
  2. 拔丝地瓜 - 45.8%
  20. 鱼香肉丝 - 40.8%
  17. 口水鸡 - 41.6%


需要多逻辑判断返回结果!!!